# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/YomnaImad07/FlyRank-ML-Internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** One row = one content item, for one client, on one specific date (a content-day row) in `fact_content_daily_performance`.

**Time window:** A single mid-panel month, `report_date` in month=2026-03, chosen to iterate safely away from the sealed final test month (June 2026, the `_sample` table).

In [20]:
import duckdb
from google.colab import userdata
import os

hf_token = userdata.get('HF_TOKEN')
os.environ["HF_TOKEN"] = hf_token

con = duckdb.connect()
con.sql("INSTALL httpfs; LOAD httpfs;")
con.sql(f"CREATE SECRET hf_token (TYPE HUGGINGFACE, TOKEN '{hf_token}');")

verify_unit_and_window = con.sql("""
SELECT
  MIN(report_date) as min_date,
  MAX(report_date) as max_date,
  COUNT(*) as total_rows,
  COUNT(DISTINCT content_hash_id) as unique_content_items,
  COUNT(DISTINCT client_hash_id) as unique_clients
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(verify_unit_and_window)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    min_date   max_date  total_rows  unique_content_items  unique_clients
0 2026-03-01 2026-03-31     9841378                331437              55


Verified: the March 2026 slice spans 2026-03-01 to 2026-03-31 as stated, with 9,841,378 rows covering 331,437 unique content items across 55 unique clients — confirming the row grain is one content item, per client, per day, not an aggregated view.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Feature (knowable before prediction, safe to use):**
- `gsc_impressions`, `gsc_clicks`, `gsc_avg_position`, `ga4_sessions`, `ga4_engaged_sessions` — observed same-day signals recorded once the day closes.

**Label / proxy (never a feature):**
- No formal label is built in this notebook. If extended, any trend/decline label derived from these same metrics (e.g., a future change in `gsc_impressions` or `gsc_avg_position`) would count as label-only, per the flyrank-data label trap warning — never mixed back in as a feature.

**Context (grouping/joining only, never learned from):**
- `content_hash_id`, `client_hash_id`, `report_date`, `month` — pseudonymous or index fields used only to group, join, and filter.

**Excluded (with reason):**
- `client_has_gsc`, `client_has_ga4`, `gsc_data_available` — availability flags kept as filters only, not features, since they describe tracking setup rather than content performance.
- Any FlyRank product decision fields (`health_score`, `priority_score`, `action_type`) — not present in this table at all, but named here as a standing exclusion per the lane guide's circular-result warning.

In [21]:
field_check = con.sql("""
SELECT
  DISTINCT gsc_data_available, ga4_data_available
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 10
""").df()
print(field_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   gsc_data_available  ga4_data_available
0                True                <NA>
1               False                <NA>
2               False               False
3                True               False
4               False                True
5                True                True


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [22]:
grain_check = con.sql("""
SELECT report_date, client_hash_id, content_hash_id, COUNT(*) as c
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING c > 1
LIMIT 5
""").df()
print("Grain check (should be empty):")
print(grain_check)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Grain check (should be empty):
Empty DataFrame
Columns: [report_date, client_hash_id, content_hash_id, c]
Index: []


**Grain check:** Grouping by `report_date + client_hash_id + content_hash_id` and filtering for groups with more than one row returns an empty result — confirming the grain holds: each row truly is one unique content item, for one client, on one date, with no duplicates.

In [23]:
count_span = con.sql("""
SELECT COUNT(*) as total_rows, MIN(report_date) as min_date, MAX(report_date) as max_date
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(count_span)

   total_rows   min_date   max_date
0     9841378 2026-03-01 2026-03-31


In [24]:
na_check = con.sql("""
SELECT
  COUNT(*) as total_rows,
  SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) as ga4_null_rows,
  SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_true_rows,
  SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) as ga4_false_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(na_check)

   total_rows  ga4_null_rows  ga4_true_rows  ga4_false_rows
0     9841378      3018741.0       413966.0       6408671.0


**Availability check (filtered with IS TRUE):** Out of 9,841,378 total rows in March 2026, only 413,966 rows (~4.2%) have `ga4_data_available IS TRUE`. Meanwhile 6,408,671 rows (~65.1%) explicitly have `ga4_data_available IS FALSE`, and 3,018,741 rows (~30.7%) have it as NULL — meaning the flag itself was never set for those rows, likely because GA4 tracking hadn't started yet for that client. This confirms that GA4-based features must be filtered with `IS TRUE` specifically, not just checked for truthiness, since NULL and FALSE both mean "don't use this row for GA4 features" but for different underlying reasons.

In [25]:
features_df = con.sql("""
SELECT content_hash_id, client_hash_id, report_date,
       gsc_impressions, gsc_clicks, gsc_avg_position, ga4_sessions
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
LIMIT 1000
""").df()
print(features_df.head())

            content_hash_id           client_hash_id report_date  \
0  content_b7e512995f79d5a6  client_73cda7b4e4f265ea  2026-03-01   
1  content_05597932fe4da067  client_73cda7b4e4f265ea  2026-03-01   
2  content_7a105f548d9c6916  client_73cda7b4e4f265ea  2026-03-01   
3  content_905aa32a0230694e  client_73cda7b4e4f265ea  2026-03-01   
4  content_a3ea9792f793ec72  client_73cda7b4e4f265ea  2026-03-01   

   gsc_impressions  gsc_clicks  gsc_avg_position  ga4_sessions  
0               20           0          3.350000          <NA>  
1                1           0          0.000000          <NA>  
2              125           1          4.928000          <NA>  
3                7           0          4.000000          <NA>  
4               11           0          2.272727          <NA>  


- `gsc_impressions`: knowable at the decision moment — observed daily search metric.
- `gsc_clicks`: knowable — directly logged by Search Console same day.
- `gsc_avg_position`: knowable — actual search ranking that day, not predicted. Note: rows with `gsc_avg_position = 0.0` (e.g., row 2 above) likely mean "no position data," consistent with the same convention flagged in the flyrank-data skill for the starter CSV — not an actual top ranking.
- `ga4_sessions`: knowable — same-day GA4 analytics count. Note: all five sample rows above show `ga4_sessions` as NULL, consistent with the earlier availability check showing only ~4.2% of rows have `ga4_data_available IS TRUE` — GA4 features must be filtered accordingly before use.
- `content_hash_id` age/context (if extended with dim_content join): knowable — static content property that exists before any ranking outcome.

In [26]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score

target = (features_df["gsc_avg_position"] <= 10).astype(int)

# STEP 1: deliberately leaky feature (built directly from the target)
features_df["leaky_echo"] = target

leaky_score = roc_auc_score(target, features_df["leaky_echo"])
print("Leaky score (suspiciously perfect):", leaky_score)

# STEP 2: remove it, use only honest, pre-decision features
honest_X = features_df[["gsc_impressions", "gsc_clicks", "ga4_sessions"]].fillna(0)
model = LogisticRegression().fit(honest_X, target)
honest_score = roc_auc_score(target, model.predict_proba(honest_X)[:, 1])
print("Honest score (after removing the leak):", honest_score)

Leaky score (suspiciously perfect): 1.0
Honest score (after removing the leak): 0.8860030690537084


Adding `leaky_echo` — a column built directly from the target itself — pushed the score to a perfect 1.0, an unmistakable red flag: no real-world model achieves perfect accuracy, and a perfect score always means the model is echoing the label rather than learning genuine signal. After removing `leaky_echo` and keeping only genuinely pre-decision features (`gsc_impressions`, `gsc_clicks`, `ga4_sessions`), the honest score dropped to a realistic 0.886 — still strong, showing these observed signals carry real predictive value on their own, without any leaked information. This confirms the leakage lesson from notebook 02, now demonstrated on real warehouse data.

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Named limitation:** This slice covers only March 2026 for clients with active tracking that month. Client history start dates vary widely (`dim_clients.gsc_data_start`), and only ~4.2% of rows have `ga4_data_available IS TRUE` — meaning GA4-based features are usable for a small minority of rows, not the full slice. Rows before a client's GA4 start show search data only (GSC-only rows), so this single month cannot fully capture engagement patterns across the whole client base. Patterns found in this single month may not generalize across the full 17-month unbalanced panel.

In [27]:
client_coverage = con.sql("""
SELECT client_hash_id, COUNT(*) as rows_this_client,
       SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
FROM read_parquet('hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet')
GROUP BY client_hash_id
ORDER BY rows_this_client DESC
LIMIT 10
""").df()
print(client_coverage)

            client_hash_id  rows_this_client  ga4_available_rows
0  client_625b6439094e23e4            988497                37.0
1  client_3ffa76342f366962            904847              4640.0
2  client_73cda7b4e4f265ea            869640             38268.0
3  client_08a6a72ff48e62c0            851275                 0.0
4  client_62f4a7e64f5e0096            756660                 0.0
5  client_65de48885f4ef01b            426307              4170.0
6  client_23a62021009f63c4            423613            146493.0
7  client_ba65e80a1116ae41            410409             10523.0
8  client_2b4306c3ed003f01            375906                 0.0
9  client_fef1a8f436438636            335379             48001.0


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.